In [16]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import torch.nn.functional as F
import math
from collections import deque
GRID_SIZE =15
BATCH_SIZE =64
GAMMA=0.99
LEARNING_RATE =1e-6
MEMORY_SIZE =100000
EPSILON_DECAY =0.99
EPSILON_START =0.05
EPSILON_END=0.1
SAVE_EP=100
TARGET_UPDATE_EP=10
EPISODES=10000
UPDATE_FREQ=4
RESET_EPSILON_EVERY=5000

In [3]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

class SimplePacmanEnv(gym.Env):
    """
    简化的吃豆人环境：
    - 网格大小 grid_size  15
    - 玩家从 (0,0) 开始
    - 随机生成约 60% 的豆子
    - 吃到豆子得 +1，吃完所有豆子得 +10，每步 -0.1
    - 状态: 2通道 (玩家位置, 豆子分布)，每个通道 grid_size x grid_size
    """
    def __init__(self, grid_size=15):
        super().__init__()
        self.grid_size = grid_size

        # 动作空间: 0上,1下,2左,3右
        self.action_space = spaces.Discrete(4)

        # 观察空间: 2通道网格，每个像素值0或1
        self.observation_space = spaces.Box(
            low=0, high=1,
            shape=(2, grid_size, grid_size),
            dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # 玩家起始位置
        self.player_pos = [0, 0]

        # 随机生成豆子（约60%）
        total_cells = self.grid_size * self.grid_size
        num_dots = int(total_cells * 0.6)
        rng = np.random.default_rng(seed)
        indices = rng.choice(total_cells, num_dots, replace=False)
        self.dot_grid = np.zeros((self.grid_size, self.grid_size), dtype=bool)
        self.dot_grid.flat[indices] = True
        self.dot_grid[0, 0] = False  # 玩家起点无豆

        self.dots_left = np.sum(self.dot_grid)
        self.steps = 0
        return self._get_obs(), {}

    def _get_obs(self):
        # 玩家通道
        player = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)
        px, py = self.player_pos
        player[px, py] = 1.0

        # 豆子通道
        dot = self.dot_grid.astype(np.float32)

        return np.stack([player, dot], axis=0)

    def step(self, action):
        self.steps += 1
        reward = -0.05
        done = False

        # 移动玩家
        x, y = self.player_pos
        if action == 0:   # 上
            y = min(y + 1, self.grid_size - 1)
        elif action == 1: # 下
            y = max(y - 1, 0)
        elif action == 2: # 左
            x = max(x - 1, 0)
        elif action == 3: # 右
            x = min(x + 1, self.grid_size - 1)
        self.player_pos = [x, y]

        # 吃豆
        if self.dot_grid[x, y]:
            self.dot_grid[x, y] = False
            self.dots_left -= 1
            reward += 1

        # 检查是否吃完所有豆子
        if self.dots_left == 0:
            reward += 10
            done = True

        # 步数上限
        if self.steps >= 1000:
            done = True

        return self._get_obs(), reward, done, {}

    def render(self, mode='human'):
        # 简单文本渲染
        grid = [['.' for _ in range(self.grid_size)] for _ in range(self.grid_size)]
        # 标记豆子
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                if self.dot_grid[i, j]:
                    grid[i][j] = 'o'
        # 标记玩家
        px, py = self.player_pos
        grid[px][py] = 'P'
        # 打印
        for row in reversed(grid):
            print(' '.join(row))
        print(f"Steps: {self.steps}, Dots left: {self.dots_left}")

In [4]:
device =torch.device('cuda'if torch.cuda.is_available() else 'cpu')

In [5]:
class NoisyLinear(nn.Module):
    def __init__(self, in_features, out_features, std_init=0.5):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight_mu = nn.Parameter(torch.Tensor(out_features, in_features))
        self.weight_sigma = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias_mu = nn.Parameter(torch.Tensor(out_features))
        self.bias_sigma = nn.Parameter(torch.Tensor(out_features))
        self.std_init = std_init
        self.reset_parameters()
        self.register_buffer("weight_epsilon", torch.Tensor(out_features, in_features))
        self.register_buffer("bias_epsilon", torch.Tensor(out_features))

    def reset_parameters(self):
        mu_range = 1 / math.sqrt(self.in_features)
        self.weight_mu.data.uniform_(-mu_range, mu_range)
        self.weight_sigma.data.fill_(self.std_init / math.sqrt(self.in_features))
        self.bias_mu.data.uniform_(-mu_range, mu_range)
        self.bias_sigma.data.fill_(self.std_init / math.sqrt(self.out_features))

    def forward(self, x):
        if self.training:
            self.weight_epsilon.normal_()
            self.bias_epsilon.normal_()
            weight = self.weight_mu + self.weight_sigma * self.weight_epsilon
            bias = self.bias_mu + self.bias_sigma * self.bias_epsilon
        else:
            weight = self.weight_mu
            bias = self.bias_mu
        return F.linear(x, weight, bias)

In [6]:
class QNetworkDuelingCNN(nn.Module):
    """Dueling ->决斗"""
    def __init__(self,input_channels=2,grid_size=15,action_size=4):
        super().__init__()
        self.conv1 =nn.Conv2d(input_channels,32,3,padding=1)#输入、输出、卷积核大小
        self.conv2 =nn.Conv2d(32,64,3,padding=1)
        self.conv3 =nn.Conv2d(64,64,3,padding=1)
        self.flatten_dim =grid_size*grid_size*64
        self.fc1=NoisyLinear(self.flatten_dim,256)
        self.value_fc =NoisyLinear(256,1)#价值函数
        self.advantage_fc =NoisyLinear(256,action_size)#优势函数

    def forward(self,x):
        x=F.relu(self.conv1(x))
        x=F.relu(self.conv2(x))
        x=F.relu(self.conv3(x))
        x=x.view(-1,self.flatten_dim)
        x=F.relu(self.fc1(x))
        value=self.value_fc(x)
        advantage=self.advantage_fc(x)
        return value+advantage-advantage.mean(dim=1,keepdim=True)#优势函数减去优势函数的均值，避免网络波动。。。###


In [7]:
class PrioritizedReplayBuffer:
    def __init__(self,capacity=MEMORY_SIZE,alpha=0.4,beta_start=0.4):
        """alpha是优先级抽取的权重，beta是补偿的系数"""
        self.capacity = capacity
        self.alpha = alpha
        self.beta=beta_start
        self.beta_increment =0.001
        self.buffer=deque(maxlen=capacity)#状态存储
        self.priorities =deque(maxlen=capacity)#权重存储
        self.max_priority=1.0

    def push(self,state,action,reward,next_state,done):
        """将数据存储到buffer中"""
        self.buffer.append((state,action,reward,next_state,done))
        self.priorities.append(self.max_priority)
    def sample(self,batch_size):
        #抽取数据，alpha区域
        priorities=np.array(self.priorities)
        probs=priorities**self.alpha
        probs/=probs.sum()
        total=len(self.buffer)
        indices =np.random.choice(total,batch_size,p=probs)#概率抽取索引
        #计算权重，beta区域
        weights =(total * probs[indices]) ** (-self.beta)#公式为weights=(N*P(i))^(-beta)
        weights /= weights.max() #归一化权重
        self.beta =min(1.0,self.beta+self.beta_increment)

        batch=[self.buffer[idx] for idx in indices]
        state, action, reward, next_state, done = zip(*batch)
        state = np.stack(state)
        action = np.array(action)#(bacth,)
        reward = np.array(reward)
        next_state = np.stack(next_state)
        done = np.array(done, dtype=np.float32)

        return (
            torch.FloatTensor(state).to(device),
            torch.LongTensor(action).unsqueeze(1).to(device),#(batch,1)
            torch.FloatTensor(reward).unsqueeze(1).to(device),
            torch.FloatTensor(next_state).to(device),
            torch.FloatTensor(done).unsqueeze(1).to(device),
            torch.FloatTensor(weights).unsqueeze(1).to(device),
            indices#每次采样后需要根据这批样本计算出新的TD误差，更新他们在缓冲区中的优先级，因为此得返回indices
        )
    def update_priorities(self,indices,tds):#|td=q_target-q_value|
        """更新样本的优先级"""
        for idx,td in zip(indices,tds):
            priority=(abs(td)+1e-6)#1e-6防止优先级为0
            self.priorities[idx] = priority
            if priority>self.max_priority:
                self.max_priority = priority
    def __len__(self):
        return len(self.buffer)

In [8]:
class DQNAgent:
    def __init__(self, action_size):
        self.action_size = action_size
        self.policy_net = QNetworkDuelingCNN().to(device)
        self.target_net = QNetworkDuelingCNN().to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LEARNING_RATE)
        self.memory = PrioritizedReplayBuffer(MEMORY_SIZE)

    def select_action(self, state):
        self.policy_net.train()   # 启用噪声
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = self.policy_net(state)
            return q_values.argmax().item()

    def update(self):
        if len(self.memory) < BATCH_SIZE:
            return
        state, action, reward, next_state, done, weights, indices = self.memory.sample(BATCH_SIZE)

        self.policy_net.train()
        current_q = self.policy_net(state).gather(1, action)

        with torch.no_grad():
            self.target_net.eval()
            self.policy_net.eval()          # 暂时关闭噪声
            next_action = self.policy_net(next_state).argmax(dim=1, keepdim=True)
            self.policy_net.train()         # 恢复训练模式
            next_q = self.target_net(next_state).gather(1, next_action)
            target_q = reward + GAMMA * (1 - done) * next_q

        loss = (weights * (current_q - target_q) ** 2).mean()

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()

        td_errors = (current_q - target_q).detach().cpu().numpy().squeeze()
        self.memory.update_priorities(indices, np.abs(td_errors))

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

In [14]:
def train(agent,episodes):
        """训练函数"""
        env = SimplePacmanEnv(grid_size=15)
        # agent.epsilon=EPSILON_START
        # agent.epsilon_end=EPSILON_END
        # agent.epsilon_decay=EPSILON_DECAY
        agent.gamma = GAMMA
        total_dots=0
        best_avg =-float('inf')#负无穷大
        total_reward =0
        total_step=0
        for episode in range(episodes):
            state,_=env.reset()
            initial_dots = env.dots_left 
            done =False
            # if episode % RESET_EPSILON_EVERY == 0:
            #     agent.epsilon = 0.5
            while not done:
                action =agent.select_action(state)
                next_state,reward,done,_=env.step(action)
                total_reward +=reward
                agent.memory.push(state, action, reward, next_state, done)
                state = next_state
                total_step += 1

                if total_step%UPDATE_FREQ==0:
                    agent.update()
            # 计算本轮吃到的豆子数量
            dots_eaten = initial_dots - env.dots_left
            total_dots += dots_eaten

            # agent.decay_epsilon()
            if episode%TARGET_UPDATE_EP==0:
                agent.update_target()
            
            if(episode+1)% SAVE_EP==0:
                agent.policy_net.cpu()
                agent.target_net.cpu()
                torch.save(agent, 'model.pth')
                agent.policy_net.to(device)
                agent.target_net.to(device)

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                print("SAVE")

            if (episode+1)%20==0:
                avg_reward = total_reward / 20
                avg_step=total_step/20
                avg_dots = total_dots / 20
                print(f"Episode: {episode}, Avg Reward: {avg_reward:.2f}, Avg Step: {avg_step:.2f}, Avg Dots Eaten: {avg_dots:.2f}")
                if avg_reward > best_avg:
                    best_avg = avg_reward
                    torch.save(agent.policy_net.state_dict(), 'best_model.pth')
                    print("SAVE")
                total_reward = 0
                total_step = 0
                total_dots = 0
        agent.policy_net.cpu()
        agent.target_net.cpu()
        torch.save(agent, 'final_model.pth')
        print("训练完成，最终模型保存为 final_model.pth")

In [26]:
def valid(agent, episodes):
    env = SimplePacmanEnv(grid_size=15)
    agent.epsilon = 0
    agent.gamma = GAMMA          # 确保 gamma 与训练时一致（如需要）
    agent.policy_net.eval()
    agent.target_net.eval()

    total_bad = 0
    total_reward = 0
    total_steps = 0
    total_dots = 0

    for episode in range(episodes):
        state, _ = env.reset()
        initial_dots = env.dots_left   # 记录开始时剩余豆子数
        done = False
        episode_reward = 0
        episode_steps = 0

        while not done:
            action = agent.select_action(state)   # 使用 agent 原有的动作选择（epsilon=0）
            next_state, reward, done, _ = env.step(action)
            episode_reward += reward
            episode_steps += 1
            state = next_state

        dots_eaten = initial_dots - env.dots_left
        print(f"Episode {episode:3d}: Reward = {episode_reward:6.2f} | Steps = {episode_steps:4d} | Dots = {dots_eaten:3d}")
        if episode_steps>350:
            total_bad += 1

        total_reward += episode_reward
        total_steps += episode_steps
        total_dots += dots_eaten

    # 最终统计
    avg_reward = total_reward / episodes
    avg_steps = total_steps / episodes
    avg_dots = total_dots / episodes
    print("\n" + "=" * 50)
    print(f"Over {episodes} episodes:")
    print(f"  Bad episodes (reward < 80): {total_bad} ({total_bad/episodes*100:.1f}%)")
    print(f"  Average reward : {avg_reward:.2f}")
    print(f"  Average steps  : {avg_steps:.2f}")
    print(f"  Average dots   : {avg_dots:.2f}")
    print("=" * 50)

In [37]:
def valid(agent, episodes):
    env = SimplePacmanEnv(grid_size=15)
    agent.policy_net.eval()   # 确保网络处于评估模式（无噪声）
    
    total_bad = 0
    total_reward = 0
    total_dots = 0
    total_steps = 0

    for episode in range(episodes):
        state, _ = env.reset()
        initial_dots = env.dots_left
        done = False
        episode_reward = 0
        episode_steps = 0

        while not done:
            # 确定性动作选择
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                q_values = agent.policy_net(state_t)
                action = q_values.argmax().item()
            
            next_state, reward, done, _ = env.step(action)
            episode_reward += reward
            episode_steps += 1
            state = next_state

        dots_eaten = initial_dots - env.dots_left
        print(f"Episode {episode}: Reward = {episode_reward:.2f}, Steps = {episode_steps}, Dots = {dots_eaten}")

        total_reward += episode_reward
        total_dots += dots_eaten
        total_steps += episode_steps
        if episode_steps >350:
            total_bad += 1

    avg_reward = total_reward / episodes
    avg_steps = total_steps / episodes
    avg_dots = total_dots / episodes
    print(f"\nOver {episodes} episodes: Bad = {total_bad} ({total_bad/episodes*100:.1f}%)")
    print(f"Avg Reward = {avg_reward:.2f}, Avg Steps = {avg_steps:.2f}, Avg Dots = {avg_dots:.2f}")

In [30]:
# model = DQNAgent(action_size=4)
# model.policy_net.load_state_dict(torch.load('reward-model02.pth', map_location=device))
# model.update_target()
# train(model,2000)


In [38]:
model = DQNAgent(action_size=4)
model.policy_net.load_state_dict(torch.load('reward-model01.pth', map_location=device))
model.policy_net.eval()
valid(model,2000)



Episode 0: Reward = 133.10, Steps = 218, Dots = 134
Episode 1: Reward = 133.40, Steps = 212, Dots = 134
Episode 2: Reward = 134.05, Steps = 219, Dots = 135
Episode 3: Reward = 133.25, Steps = 215, Dots = 134
Episode 4: Reward = 133.65, Steps = 207, Dots = 134
Episode 5: Reward = 133.35, Steps = 213, Dots = 134
Episode 6: Reward = 134.15, Steps = 217, Dots = 135
Episode 7: Reward = 135.75, Steps = 185, Dots = 135
Episode 8: Reward = 134.65, Steps = 207, Dots = 135
Episode 9: Reward = 134.90, Steps = 202, Dots = 135
Episode 10: Reward = 134.05, Steps = 199, Dots = 134
Episode 11: Reward = 134.55, Steps = 189, Dots = 134
Episode 12: Reward = 134.25, Steps = 195, Dots = 134
Episode 13: Reward = 133.75, Steps = 205, Dots = 134
Episode 14: Reward = 134.55, Steps = 209, Dots = 135
Episode 15: Reward = 134.25, Steps = 215, Dots = 135
Episode 16: Reward = 133.90, Steps = 202, Dots = 134
Episode 17: Reward = 133.15, Steps = 217, Dots = 134
Episode 18: Reward = 134.00, Steps = 200, Dots = 134
Epi

In [35]:
def valid(agent, episodes):
    env = SimplePacmanEnv(grid_size=15)
    agent.epsilon = 0
    agent.gamma = GAMMA          # 确保 gamma 与训练时一致（如需要）
    agent.policy_net.eval()
    agent.target_net.eval()

    total_bad = 0
    total_reward = 0
    total_steps = 0
    total_dots = 0

    for episode in range(episodes):
        state, _ = env.reset()
        initial_dots = env.dots_left   # 记录开始时剩余豆子数
        done = False
        episode_reward = 0
        episode_steps = 0

        while not done:
            action = agent.select_action(state)   # 使用 agent 原有的动作选择（epsilon=0）
            next_state, reward, done, _ = env.step(action)
            episode_reward += reward
            episode_steps += 1
            state = next_state

        dots_eaten = initial_dots - env.dots_left
        print(f"Episode {episode:3d}: Reward = {episode_reward:6.2f} | Steps = {episode_steps:4d} | Dots = {dots_eaten:3d}")
        if episode_steps>350:
            total_bad += 1

        total_reward += episode_reward
        total_steps += episode_steps
        total_dots += dots_eaten

    # 最终统计
    avg_reward = total_reward / episodes
    avg_steps = total_steps / episodes
    avg_dots = total_dots / episodes
    print("\n" + "=" * 50)
    print(f"Over {episodes} episodes:")
    print(f"  Bad episodes (reward < 80): {total_bad} ({total_bad/episodes*100:.1f}%)")
    print(f"  Average reward : {avg_reward:.2f}")
    print(f"  Average steps  : {avg_steps:.2f}")
    print(f"  Average dots   : {avg_dots:.2f}")
    print("=" * 50)

In [36]:
model = DQNAgent(action_size=4)
model.policy_net.load_state_dict(torch.load('reward-model01.pth', map_location=device))
model.policy_net.eval()
valid(model,2000)


Episode   0: Reward = 134.70 | Steps =  206 | Dots = 135
Episode   1: Reward = 134.55 | Steps =  209 | Dots = 135
Episode   2: Reward = 134.50 | Steps =  210 | Dots = 135
Episode   3: Reward = 135.00 | Steps =  200 | Dots = 135
Episode   4: Reward = 133.70 | Steps =  206 | Dots = 134
Episode   5: Reward = 134.00 | Steps =  200 | Dots = 134
Episode   6: Reward = 133.95 | Steps =  201 | Dots = 134
Episode   7: Reward = 133.00 | Steps =  220 | Dots = 134
Episode   8: Reward = 134.20 | Steps =  216 | Dots = 135
Episode   9: Reward = 135.10 | Steps =  198 | Dots = 135
Episode  10: Reward = 133.80 | Steps =  204 | Dots = 134
Episode  11: Reward = 133.20 | Steps =  216 | Dots = 134
Episode  12: Reward = 133.25 | Steps =  215 | Dots = 134
Episode  13: Reward = 133.85 | Steps =  203 | Dots = 134
Episode  14: Reward = 134.10 | Steps =  198 | Dots = 134
Episode  15: Reward = 132.80 | Steps =  224 | Dots = 134
Episode  16: Reward = 135.30 | Steps =  194 | Dots = 135
Episode  17: Reward = 134.80 | 